### Hybrid Retrival Argumented Generation Evelution using RAGAS 

In [1]:
import warnings 
warnings.filterwarnings('ignore')

# Document load 
from langchain_community.document_loaders import PyPDFLoader 
loader  = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

# Split Data 

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [3]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# client and collection create 
client = chromadb.PersistentClient(path="./Hybrid_RAG")
collection = client.get_or_create_collection(name="Hybrid_RAG",embedding_function=embedding_function)

if chunks:
    collection.add(
        ids=ids,
        documents=chunks , metadatas=metadata
    )
collection.count()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

225

In [4]:
# LLM call 
import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()
try:
    key = os.getenv('GROQ_API_KEY')
    print(bool(key))
except Exception as e:
    print(str(e))
    
Groq = ChatGroq(model="qwen/qwen3.6-27b")

test = Groq.invoke("hello llama?")
test.content

True


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "hello llama?"\n   - This is a greeting, possibly referencing "Llama" (the AI model family by Meta), but I need to clarify my identity.\n\n2.  **Identify Key Facts:**\n   - I am Qwen (通义千问), developed by Alibaba Group\'s Tongyi Lab.\n   - I should respond politely, acknowledge the greeting, and clarify my identity if needed.\n   - Keep it friendly and concise.\n\n3.  **Formulate Response:**\n   - Acknowledge the greeting: "Hello!"\n   - Clarify identity politely: "I\'m actually Qwen, not Llama. How can I assist you today?"\n   - Keep it open-ended to encourage the user\'s next question.\n\n4.  **Self-Correction/Refinement:**\n   - Check tone: Friendly, helpful, accurate.\n   - Avoid unnecessary details.\n   - Ensure alignment with identity guidelines.\n   - The response matches all criteria.\n\n   Final version: "Hello! I\'m actually Qwen, not Llama. How can I help you today?"✅\n</think>\n\nHello! I

In [5]:
# Hybrid Corpus 
from rank_bm25 import BM25Okapi 
def tokenization(token):
    token = token.lower()
    token = token.split()
    return token 

tokens = [tokenization(i) for i in chunks]
bm_corpus = BM25Okapi(tokens)

print(f'sucussfully : {bm_corpus}')

sucussfully : <rank_bm25.BM25Okapi object at 0x129e9a3f0>


In [6]:
def Hybrid_Retrive(query:str):
    query_re = Groq.invoke(f"write the query based on symentic search : {query}").content.strip()

    # Thats Vector DB retrival 
    result = collection.query(query_texts=[query_re] , n_results=5)
    dis  = result['distances'][0] 
    docs = result['documents'][0]
    threshold = 0.9
    print(f'the distance is : {dis}')
    dense_docs = []
    for i , d in zip(dis,docs):
        if threshold > i :
            dense_docs.append(d)
    # Thats Hybrid RAG Retrival using indexing 
    query_tokens = tokenization(query_re)
    score = bm_corpus.get_scores(query=query_tokens)
    def get_top_tokens (score , k=10):
        index = list(enumerate(score))
        idx_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [doc for doc , _ in idx_sorted[:10]]
    index_tokens = [chunks[i] for i in get_top_tokens(score=score,k=10)]
    
    rrf_token = {}
    
    for rank , doc in enumerate(dense_docs):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_tokens):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
        
    marge = sorted(rrf_token.items() , key = lambda x:x[1] , reverse=True)
    get_docs = [i for i , _ in marge[:5]]
    
    return get_docs
    
def generation_answer(question:str , context_list:list):
    if not context_list :
        return "NOT Related Content"
    content_str = "\n\n".join(context_list) 
    
    prompt = f""" 
    Give answer based on the local document , if cant find out any related content 
    then direct type NOT related content 
    content : {content_str}
    question :{question}
    """
    response = Groq.invoke(prompt)
    
    return response.content

In [8]:
# RAG evelute 
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from langchain_community.embeddings import HuggingFaceEmbeddings 

user_input = []
retrival_context = []
response = []
reference = []

test_cases = [
    {
        "question": "who is first First Chief of Army Staff",
        "ground_truth": "General Maharaj Rajendra Singh Ji was the first Chief of Army Staff."
    },
    {
        "question": "Where is Malhargad Fort located?",
        "ground_truth": "Malhargad Fort is located in Sonori, near Saswad, Pune district, Maharashtra."
    },
    {
        "question": "Who built the Red Fort in Delhi?",
        "ground_truth": "Red Fort was built by Mughal Emperor Shah Jahan in 1648 AD."
    },
    {
        "question": "What is Purandar Fort famous for?",
        "ground_truth": "Purandar Fort is famous as the birthplace of Chhatrapati Sambhaji Maharaj."
    },
    {
        "question": "Which dynasty built Chitradurga Fort originally?",
        "ground_truth": "Chitradurga Fort was originally built by the Chalukyas between the 11th and 13th centuries."
    }
]
for item in test_cases:
    q =  item['question']
    truth = item['ground_truth']
    
    context = Hybrid_Retrive(query=q)
    LLM_answer = generation_answer(question=q , context_list=context)
    
    user_input.append(q)
    retrival_context.append(context)
    response.append(LLM_answer)
    reference.append(truth)
    
    data = {
        "user_input":user_input , 
        "retrieved_contexts":retrival_context , 
        "response":response , 
        "reference":reference
    }
    data = Dataset.from_dict(data)

    embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print(LLM_answer)

result = evaluate(
dataset= data, 
metrics=[Faithfulness(),AnswerRelevancy(),ContextPrecision(),ContextRecall()],
embeddings=embedding , 
llm=Groq,
raise_exceptions=False
)

df = result.to_pandas()
print(df)

the distance is : [0.6230571866035461, 0.7254743576049805, 0.7291903495788574, 0.7349518537521362, 0.7350471615791321]


/var/folders/j2/t5w7lcnj6wg9gmp3h6zxzlk00000gn/T/ipykernel_1379/3345069561.py:54: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the distance is : [0.6668291091918945, 0.7127556800842285, 0.7162514328956604, 0.7249287366867065, 0.7251556515693665]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Which dynasty built Chitradurga Fort originally?"
   - **Constraint:** "Give answer based on the local document, if cant find out any related content then direct type NOT related content"
   - **Provided Document:** Contains multiple pages of a "GK Now – Current Affairs" document listing various forts, their locations, builders, years, and notes. I need to scan the document for "Chitradurga Fort".

2.  **Scan Document for Keywords:**
   - Keyword: "Chitradurga Fort"
   - Found in the first block of text:
     ```
     Chitradurga Fort Chitradurga, 
     Karnataka Chalukyas 11th and 
     13th Centuries 
     Chitradurga Fort or as 
     the British called it 
     Chitaldoorg, is a 
     fortification that 
     straddles several hills 
     and a peak 
     overlooking a flat 
     valley in the 
     Chitradurga District, 
     Karnataka, India. The 
     fort was built in 
     stages between the 
 

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[4]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[1]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[5]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[13]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception 

                                         user_input  \
0            who is first First Chief of Army Staff   
1                  Where is Malhargad Fort located?   
2                  Who built the Red Fort in Delhi?   
3                 What is Purandar Fort famous for?   
4  Which dynasty built Chitradurga Fort originally?   

                                  retrieved_contexts  \
0  [GK Now – Current Affairs \n12 \nGK Now – Curr...   
1  [GK Now – Current Affairs \n155 \nGK Now – Cur...   
2  [GK Now – Current Affairs \n162 \nGK Now – Cur...   
3  [GK Now – Current Affairs \n156 \nGK Now – Cur...   
4  [GK Now – Current Affairs \n160 \nGK Now – Cur...   

                                            response  \
0  \n<think>\nHere's a thinking process:\n\n1.  *...   
1  \n<think>\nHere's a thinking process:\n\n1.  *...   
2  \n<think>\nHere's a thinking process:\n\n1.  *...   
3  \n<think>\nHere's a thinking process:\n\n1.  *...   
4  \n<think>\nHere's a thinking process:\n\n1.  *...